## BERT Fine-Tuning

This notebook fine-tunes a pretrained BERT model on the dataset for multi-class text classification and evaluates its performance. The model is trained end-to-end, allowing contextual embeddings and classification layers to be optimized jointly for the task.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

In [3]:
train_df = pd.read_parquet(r"/content/drive/MyDrive/Colab Notebooks/train.parquet")
test_df  = pd.read_parquet(r"/content/drive/MyDrive/Colab Notebooks/test.parquet")

In [4]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
bert = AutoModel.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
train_set = train_df["combined_context"].tolist()
test_set = test_df["combined_context"].tolist()
y_train  = train_df["final_decision"].tolist()
y_test   =  test_df["final_decision"].tolist()


In [6]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len = 256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(text, padding = "max_length", truncation = True, max_length = self.max_len, return_tensors = "pt")

        return {"input_ids": encoding["input_ids"].squeeze(0),
                "attention_mask": encoding["attention_mask"].squeeze(0),
                "label": torch.tensor(label, dtype = torch.long)}

In [7]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [8]:
train_dataset = TextDataset(train_set, y_train, tokenizer)
test_dataset = TextDataset(test_set, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=False)

In [9]:
class BERTClassifier(nn.Module):
    def __init__(self, bert, num_classes):
        super().__init__()
        self.bert = bert
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(768, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids = input_ids, attention_mask = attention_mask)
        cls_output = outputs.last_hidden_state[:,0,:]
        x = self.dropout(cls_output)
        return self.fc(x)

In [10]:
num_classes = 3
model = BERTClassifier(bert, num_classes)

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [16]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = model.to(device)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:

        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch + 1}/{epochs}, Loss: {avg_loss:.4f}")

Using device: cuda
Epoch 1/10, Loss: 0.2145
Epoch 2/10, Loss: 0.1088
Epoch 3/10, Loss: 0.0274
Epoch 4/10, Loss: 0.0311
Epoch 5/10, Loss: 0.0122
Epoch 6/10, Loss: 0.1148
Epoch 7/10, Loss: 0.0079
Epoch 8/10, Loss: 0.0038
Epoch 9/10, Loss: 0.0027
Epoch 10/10, Loss: 0.0021


In [17]:
model.eval()

preds = []
true = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask)

        predictions = torch.argmax(outputs, dim=1)

        preds.extend(predictions.cpu().numpy())
        true.extend(labels.cpu().numpy())

In [18]:
from sklearn.metrics import classification_report
true = le.inverse_transform(true)
preds = le.inverse_transform(preds)

print(classification_report(true, preds))

              precision    recall  f1-score   support

       maybe       0.00      0.00      0.00        11
          no       0.47      0.44      0.45        34
         yes       0.62      0.75      0.68        55

    accuracy                           0.56       100
   macro avg       0.36      0.40      0.38       100
weighted avg       0.50      0.56      0.53       100

